# Notebook 03 — Pipeline Dry Run

End-to-end dry run of **Layer 1 + Layer 2** on 5 Tier 1 prompts. No 3D generation. Validates that graph retrieval → scene spec generation works before committing GPU time.

**Set your `ANTHROPIC_API_KEY` in Cell 1 before running.**

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
import os, sys
REPO = 'ifc-graphrag-dt'

if os.path.exists(REPO):
    !cd {REPO} && git pull --quiet
else:
    !git clone https://github.com/aiwithprashant/ifc-graphrag-dt.git

!bash {REPO}/colab_setup.sh

os.chdir(REPO)
REPO = '.'   # paths after chdir must not include repo name
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# ── Set API key (use Colab Secrets for repeated use) ──────────────────────────
# Option A: Colab Secrets (recommended — left sidebar → key icon)
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print('API key loaded from Colab Secrets.')
except Exception:
    # Option B: paste directly (do not commit)
    os.environ['ANTHROPIC_API_KEY'] = 'your-key-here'
    print('Using inline API key — replace with your actual key.')

print(f'Working directory: {os.getcwd()}')

import json
from pathlib import Path
from pipeline.layer1_retriever.ifc_graph_builder import IFCGraphBuilder
from pipeline.layer1_retriever.graph_embedder import GraphEmbedder
from pipeline.layer1_retriever.khop_traversal import KHopTraversal
from pipeline.layer2_spec_gen.scene_spec_generator import SceneSpecGenerator
from evaluation.stage_a.spec_evaluator import StageAEvaluator
from benchmark.dtah_bench import DTAHBench

IFC_PATH    = 'benchmark/ifc_reference_models/duplex.ifc'
GRAPH_CACHE = 'outputs/graphs/ifc_graph.json'
EMB_CACHE   = 'outputs/embedders/graph_embedder'
SPEC_OUT    = 'outputs/specs'
os.makedirs(SPEC_OUT, exist_ok=True)
os.makedirs('outputs/scores', exist_ok=True)
print('Imports OK')

In [ ]:
# ── Cell 2: Load graph and embedder ──────────────────────────────────────────
if not Path(IFC_PATH).exists():
    import urllib.request
    urllib.request.urlretrieve(
        'https://github.com/buildingSMART/Sample-Test-Files/raw/master/IFC%202x3/Duplex%20Apartment/Duplex_A_20110907.ifc',
        IFC_PATH)
    print('duplex.ifc downloaded')

if Path(GRAPH_CACHE).exists():
    G = IFCGraphBuilder.load_graph(GRAPH_CACHE)
else:
    builder = IFCGraphBuilder(IFC_PATH)
    G = builder.build()
    builder.save_graph(GRAPH_CACHE)

if Path(f'{EMB_CACHE}/faiss.index').exists():
    embedder = GraphEmbedder.load(EMB_CACHE)
else:
    print('Building embedder (2-5 min on CPU)...')
    embedder = GraphEmbedder(model_name='sentence-transformers/all-MiniLM-L6-v2')
    embedder.fit(G)
    embedder.save(EMB_CACHE)

print(f'Graph: {G.number_of_nodes()} nodes | Embedder: {len(embedder._node_ids)} nodes')

In [ ]:
# ── Cell 3: Initialise Layer 2 spec generator ─────────────────────────────────
spec_config = {
    'llm_provider': 'anthropic',
    'model':        'claude-sonnet-4-20250514',
    'max_tokens':   2048,
    'temperature':  0.0,
    'output_dir':   SPEC_OUT,
}
spec_gen  = SceneSpecGenerator(spec_config)
evaluator = StageAEvaluator()
print('SceneSpecGenerator initialised with Anthropic Claude.')

In [ ]:
# ── Cell 4: Dry run — 5 Tier 1 prompts ───────────────────────────────────────
bench   = DTAHBench(pilot_mode=True)
tier1   = bench.load_tier(1)[:5]
results = []

for p in tier1:
    pid, prompt, tier, domain = p['id'], p['prompt'], p['tier'], p.get('domain','MEP')
    print(f'\n[{pid}] {prompt}')

    # Layer 1
    seeds    = embedder.retrieve_seeds(prompt, top_k=5)
    seed_ids = [s['node_id'] for s in seeds]
    trav     = KHopTraversal(G, max_depth=1, bidirectional=True)  # Tier 1 → depth=1
    sg       = trav.traverse(seed_ids).subgraph
    print(f'  Layer 1: {sg.number_of_nodes()} nodes, {sg.number_of_edges()} edges')

    # Layer 2
    spec = spec_gen.generate(prompt=prompt, subgraph=sg,
                             prompt_id=pid, tier=tier, domain=domain)
    print(f'  Layer 2: {len(spec.get("entities",[]))} entities, '
          f'{len(spec.get("relations",[]))} relations')
    print(f'  SD prompt: {spec.get("generation_prompt","")[:80]}...')

    results.append({'id': pid, 'prompt': prompt, 'spec': spec,
                    'nodes': sg.number_of_nodes(), 'edges': sg.number_of_edges()})

print(f'\n✓ Dry run complete: {len(results)} prompts processed')

In [ ]:
# ── Cell 5: Inspect first spec ────────────────────────────────────────────────
print(f'Spec for: {results[0]["id"]} — {results[0]["prompt"]}')
print('='*60)
display_spec = {k: v for k, v in results[0]['spec'].items() if k != 'generation_prompt'}
print(json.dumps(display_spec, indent=2))

In [ ]:
# ── Cell 6: Stage A self-consistency check ────────────────────────────────────
print('Stage A self-consistency (pred vs self — should be ~1.0):')
print(f'{"ID":<15} {"E":>6} {"R":>6} {"A":>6} {"Total":>8}')
print('-' * 45)
for r in results:
    s = evaluator.evaluate(r['spec'], r['spec'], prompt_id=r['id'])
    print(f'{r["id"]:<15} {s.entity_score:>6.3f} {s.relation_score:>6.3f} '
          f'{s.attribute_score:>6.3f} {s.total:>8.3f}')
print('\nNote: Real Stage A scores require annotated ground-truth specs.')

In [ ]:
# ── Cell 7: Save dry run summary ─────────────────────────────────────────────
summary = [{'id': r['id'], 'prompt': r['prompt'],
            'subgraph_nodes': r['nodes'], 'subgraph_edges': r['edges'],
            'spec_entities': len(r['spec'].get('entities',[])),
            'spec_relations': len(r['spec'].get('relations',[])),
            'spec_constraints': len(r['spec'].get('constraints',[])),
           } for r in results]

out = 'outputs/scores/dry_run_summary.json'
with open(out, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Summary saved: {out}')
print(json.dumps(summary, indent=2))
print('\nDry run complete. Proceed to Notebook 04 (GPU required).')